In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_Pusa, Delhi - DPCC.xlsx",skiprows=16)

In [3]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,CO,Ozone,Benzene,Toluene,RH,WS,WD,SR,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,180.39,321.27,77.07,56.91,91.91,1.22,5.25,4.34,20.19,81.56,0.30,270.61,64.85,985.40,13.04,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,174.54,305.42,62.35,45.51,73.80,1.32,5.46,4.26,18.90,83.20,0.30,254.18,60.84,985.14,13.16,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,156.71,272.60,61.20,21.05,59.69,0.73,4.82,5.14,19.18,95.00,0.30,165.17,6.77,984.34,11.49,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,304.33,420.41,8.46,89.42,53.43,0.82,23.04,5.66,24.44,81.37,0.30,171.67,93.51,984.35,15.14,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,141.62,247.79,13.69,43.59,33.37,0.77,8.95,2.96,6.92,85.70,0.30,175.77,75.43,984.15,13.46,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,304.06,496.05,78.37,104.84,119.64,1.94,18.43,7.59,17.53,63.72,0.45,147.06,91.76,986.49,16.09,0.0,0.0
316,13-11-2025 00:00,14-11-2025 00:00,279.67,438.74,92.74,104.19,131.43,1.99,14.20,6.24,17.69,67.44,0.41,172.44,86.01,986.97,16.10,0.0,0.0
317,14-11-2025 00:00,15-11-2025 00:00,238.88,405.88,82.38,104.72,122.54,1.92,13.66,6.32,17.33,66.70,0.32,289.54,86.70,986.76,16.10,0.0,0.0
318,15-11-2025 00:00,16-11-2025 00:00,226.33,383.04,83.48,101.96,122.12,1.87,14.82,6.07,17.05,66.92,0.34,247.53,90.21,987.41,16.10,0.0,0.0


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (320, 19)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): []
Dropped rows (>70% NaN): 0
Missing values after imputation:
 From Date    0
To Date      0
PM2.5        0
PM10         0
NO           0
NO2          0
NOx          0
CO           0
Ozone        0
Benzene      0
Toluene      0
RH           0
WS           0
WD           0
SR           0
BP           0
AT           0
RF           0
TOT-RF       0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:

# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (320, 19)
          From Date           To Date   PM2.5    PM10     NO    NO2    NOx  \
0  01-01-2025 00:00  02-01-2025 00:00   51.59  321.27  77.07  56.91  91.91   
1  02-01-2025 00:00  03-01-2025 00:00   51.59  305.42  62.35  45.51  73.80   
2  03-01-2025 00:00  04-01-2025 00:00   51.59  272.60  61.20  21.05  59.69   
3  04-01-2025 00:00  05-01-2025 00:00   51.59  166.27   8.46  89.42  53.43   
4  05-01-2025 00:00  06-01-2025 00:00  141.62  247.79  13.69  43.59  33.37   

     CO  Ozone  Benzene  Toluene     RH   WS      WD     SR      BP     AT  \
0  1.22   5.25     4.34    20.19  81.56  0.3  270.61  64.85  985.40  13.04   
1  1.32   5.46     4.26    18.90  83.20  0.3  254.18  60.84  985.14  13.16   
2  0.73   4.82     5.14    19.18  95.00  0.3  165.17   6.77  984.34  11.49   
3  0.82  23.04     5.66    24.44  81.37  0.3  171.67  93.51  984.35  15.14   
4  0.77   8.95     2.96     6.92  85.70  0.3  175.77  75.43  984.15  13.46   

    RF  TOT-RF  
0  0.0     0.0  
1  0.

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df.to_excel('Pusa2025.xlsx', index=False)